### In this notebook we will build meaningful SQL Analysis
Query structure:
1-Title
2-Goal
3-Results
4-Interpretation

In [1]:
import sqlite3
%load_ext sql

In [2]:
con=sqlite3.connect('database/chicago.db')
%sql sqlite:///database/chicago.db

## CRIMES table

#### Query1: Top Crimes Types
Identify which crime categories are most frequent in Chicago

In [3]:
%%sql
SELECT PRIMARY_TYPE, COUNT(*) AS incident_count 
FROM crimes_data 
GROUP BY PRIMARY_TYPE ORDER BY COUNT(*) DESC LIMIT 10;

 * sqlite:///database/chicago.db
Done.


PRIMARY_TYPE,incident_count
THEFT,106
BATTERY,92
CRIMINAL DAMAGE,58
NARCOTICS,54
OTHER OFFENSE,32
ASSAULT,32
BURGLARY,30
MOTOR VEHICLE THEFT,24
ROBBERY,20
DECEPTIVE PRACTICE,20


If crimes like THEFT, BATTERY OR CRIMINAL DAMAGE rank highest, this indicates priority areas for public safety interventions.

#### Query2:Crimes Involving Arrests (Effectiveness of Policing)
Measure the percentage of crimes that result in an arrest.

In [8]:
%%sql
SELECT PRIMARY_TYPE, COUNT(*) AS incident_count,
SUM(ARREST) AS total_arrests,
ROUND((SUM(ARREST)*100)/COUNT(*),2) AS arrest_rate
FROM crimes_data
GROUP BY PRIMARY_TYPE ORDER BY arrest_rate LIMIT 15;

 * sqlite:///database/chicago.db
Done.


PRIMARY_TYPE,incident_count,total_arrests,arrest_rate
ARSON,2,0,0.0
HUMAN TRAFFICKING,1,0,0.0
INTIMIDATION,1,0,0.0
KIDNAPPING,1,0,0.0
NON - CRIMINAL,1,0,0.0
NON-CRIMINAL,1,0,0.0
NON-CRIMINAL (SUBJECT SPECIFIED),1,0,0.0
RITUALISM,1,0,0.0
STALKING,1,0,0.0
BURGLARY,30,1,3.0


A high arrest rate indicates crimes that are easier to investigate while a low arrest rate may suggest complexity or lack of evidence.This provides insights into policing efficiency

#### Query3: Crimes in the Most Dangerous Districts
Find which police districts report the most crime

In [15]:
%%sql
SELECT
    DISTRICT,
    PRIMARY_TYPE,
    COUNT(*) AS crime_count
FROM crimes_data
GROUP BY DISTRICT
ORDER BY crime_count
LIMIT 10;

 * sqlite:///database/chicago.db
Done.


DISTRICT,PRIMARY_TYPE,crime_count
20,THEFT,9
22,THEFT,10
24,THEFT,10
1,THEFT,14
5,THEFT,16
14,THEFT,17
17,THEFT,17
18,THEFT,17
16,THEFT,19
2,THEFT,21


This reveals the districts with the highest crime concentration

## socio_data Table

#### Query1: Highest Poverty Area
Identify which community area have the highest poverty rate

In [16]:
%%sql
SELECT COMMUNITY_AREA_NAME, PERCENT_HOUSEHOLDS_BELOW_POVERTY
FROM socio_data
GROUP BY COMMUNITY_AREA_NAME
ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC LIMIT 10;

 * sqlite:///database/chicago.db
Done.


COMMUNITY_AREA_NAME,PERCENT_HOUSEHOLDS_BELOW_POVERTY
Riverdale,56.5
Fuller Park,51.2
Englewood,46.6
North Lawndale,43.1
East Garfield Park,42.4
Washington Park,42.1
West Garfield Park,41.7
Armour Square,40.1
Oakland,39.7
West Englewood,34.4


This highlights the most vulnerable neighborhoods economically. These areas often face high crime, low school performance, and reduced opportunities.We will later join this with crimes and schools tables.

#### Query 2: Relationship between Income and Hardship
Check wether low income corresponds to high hardship.

In [18]:
%%sql
SELECT COMMUNITY_AREA_NAME, PER_CAPITA_INCOME, HARDSHIP_INDEX
FROM socio_data
ORDER BY HARDSHIP_INDEX DESC LIMIT 20;

 * sqlite:///database/chicago.db
Done.


COMMUNITY_AREA_NAME,PER_CAPITA_INCOME,HARDSHIP_INDEX
Riverdale,8201,98.0
Fuller Park,10432,97.0
South Lawndale,10402,96.0
Englewood,11888,94.0
Gage Park,12171,93.0
West Garfield Park,10934,92.0
New City,12765,91.0
West Englewood,11317,89.0
Washington Park,13785,88.0
North Lawndale,12034,87.0


Communities with low income and high hardship index face severe socioeconomic challenges. 

#### Query 3: Area with High Unemployment
List communities with the highest unemployment levels

In [21]:
%%sql
SELECT COMMUNITY_AREA_NAME, PERCENT_AGED_16__UNEMPLOYED
FROM socio_data
ORDER BY PERCENT_AGED_16__UNEMPLOYED DESC LIMIT 10;

 * sqlite:///database/chicago.db
Done.


COMMUNITY_AREA_NAME,PERCENT_AGED_16__UNEMPLOYED
West Englewood,35.9
Riverdale,34.6
Fuller Park,33.9
Oakland,28.7
Washington Park,28.6
Auburn Gresham,28.3
Englewood,28.0
West Garfield Park,25.8
Grand Boulevard,24.3
Chatham,24.0


Unemployment is a known risk factor for crime and poor educational outcomes. This query highlights communities that may require economic intervention.

## schools_data Table

#### Query 1: Schools with the Highest Safety Score
Identify the safetest schools in Chicago

In [23]:
%%sql
SELECT NAME_OF_SCHOOL, SAFETY_SCORE
FROM schools_data
ORDER BY SAFETY_SCORE DESC
LIMIT 10;

 * sqlite:///database/chicago.db
Done.


NAME_OF_SCHOOL,SAFETY_SCORE
Abraham Lincoln Elementary School,99.0
Alexander Graham Bell Elementary School,99.0
Annie Keller Elementary Gifted Magnet School,99.0
Augustus H Burley Elementary School,99.0
Edgar Allan Poe Elementary Classical School,99.0
Edgebrook Elementary School,99.0
Ellen Mitchell Elementary School,99.0
James E McDade Elementary Classical School,99.0
James G Blaine Elementary School,99.0
LaSalle Elementary Language Academy,99.0


This query highlights schools with excellent safety score indicators. It allows comparison with crime levels in the same community area(in future join).

#### Query 3: Schools with the Lowest graduation Rate
Identify schools where students graduate at very low rates.

In [30]:
%%sql
SELECT NAME_OF_SCHOOL, Graduation_Rate__
FROM schools_data
ORDER BY Graduation_Rate__ 
LIMIT 10;

 * sqlite:///database/chicago.db
Done.


NAME_OF_SCHOOL,Graduation_Rate__
Northside Learning Center High School,0
Ray Graham Training Center High School,10.3
Dyett High School,33.7
Christian Fenger Academy High School,34.8
Emil G Hirsch Metropolitan High School,36.2
High School of Leadership at South Shore,36.6
John Marshall Metropolitan High School,36.8
William Rainey Harper High School,37
Roger C Sullivan High School,39.3
Nicholas Senn High School,40.1


Low graduation rates indicate struggling school environments. This query helps identify performance gaps and communities needing help.

#### Query 3: Relationship between School Performance and Misconduct
Check if low performing schools have high misconduct rates

In [27]:
%%sql 
SELECT NAME_OF_SCHOOL, CPS_Performance_Policy_Level, Rate_of_Misconducts__per_100_students_
FROM schools_data
ORDER BY Rate_of_Misconducts__per_100_students_ DESC
LIMIT 10;

 * sqlite:///database/chicago.db
Done.


NAME_OF_SCHOOL,CPS_Performance_Policy_Level,Rate_of_Misconducts__per_100_students_
Moses Montefiore Special Elementary School,Level 3,251.6
William T Sherman Elementary School,Level 3,230.6
Alexander von Humboldt Elementary School,Level 3,185.5
Florence B Price Elementary School,Level 3,156.6
Daniel R Cameron Elementary School,Level 3,153.9
Oliver Wendell Holmes Elementary School,Level 3,139.8
James R Doolittle Jr Elementary School,Level 3,134.8
Kate S Buckingham Special Education Center,Level 3,116.9
Julia C Lathrop Elementary School,Level 3,113.5
Enrico Fermi Elementary School,Level 3,101.2


Schools with high misconduct rates often have lower academic performance

## JOIN Queries

#### 1- Crime Count + Hardship Communities face more crime
Identify whether high hardship communities face more crime.

In [32]:
%%sql
SELECT sd.COMMUNITY_AREA_NAME, sd.HARDSHIP_INDEX, 
(SELECT COUNT(*) FROM crimes_data AS c WHERE c.COMMUNITY_AREA_NUMBER=sd.COMMUNITY_AREA_NUMBER) AS total_crimes
FROM socio_data AS sd
ORDER BY total_crimes DESC
LIMIT 10;

 * sqlite:///database/chicago.db
Done.


COMMUNITY_AREA_NAME,HARDSHIP_INDEX,total_crimes
Austin,73.0,43
Humboldt park,85.0,22
Englewood,94.0,21
Near West Side,15.0,16
North Lawndale,87.0,16
Near North Side,1.0,15
Auburn Gresham,74.0,14
West Town,10.0,13
Chicago Lawn,80.0,12
West Englewood,89.0,12


Shows which communities have both high crime volume and high hardship index. Useful to detect socio economic vulnerability.

#### 2- Poverty vs Crime 
Do communities which more poverty experience more crime?

In [34]:
%%sql
SELECT sd.COMMUNITY_AREA_NAME, sd.PERCENT_HOUSEHOLDS_BELOW_POVERTY,
(SELECT COUNT(*) FROM crimes_data AS c WHERE c.COMMUNITY_AREA_NUMBER=sd.COMMUNITY_AREA_NUMBER) AS total_crimes
FROM socio_data AS sd
ORDER BY sd.PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC
LIMIT 10;

 * sqlite:///database/chicago.db
Done.


COMMUNITY_AREA_NAME,PERCENT_HOUSEHOLDS_BELOW_POVERTY,total_crimes
Riverdale,56.5,2
Fuller Park,51.2,2
Englewood,46.6,21
North Lawndale,43.1,16
East Garfield Park,42.4,8
Washington Park,42.1,0
West Garfield Park,41.7,10
Armour Square,40.1,0
Oakland,39.7,0
West Englewood,34.4,12


Lists communities with very high poverty and their crime volume. Helps understand economic pressure vs crime